In [ ]:
# !pip install segmentation-models-pytorch

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached click-8.3.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached rich-14.3.3-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/642.6 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/642.6 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/642.6 kB ? eta -:--:--
   ------------


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
import cv2
import os
import numpy as np
from torch.utils.data import Dataset,DataLoader, random_split
import segmentation_models_pytorch as smp

d:\Users\Asus\Liver_Tumor_Segmentation_Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class LiverDataset(Dataset):
    def __init__(self,img_dir,mask_dir):
        self.images = os.listdir(img_dir)
        self.img_dir = img_dir
        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.images)
    
    def __getitem__(self,idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask__dir, img_name.replace("img","mask"))

        img = cv2.imread(img_path,0)
        mask = cv2.imread(mask_path,0)

        img = torch.tensor(img).unsqueeze(0).float()/255.0
        mask = torch.tensor(mask).long()

        return img, mask

In [3]:
dataset = LiverDataset("../data/images","../data/masks")
print("Total samples : ",len(dataset))

Total samples :  3067


In [4]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [5]:
train_loader = DataLoader(train_dataset, batch_size = 8, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 8)

### Model

In [6]:
model = smp.DeepLabV3Plus(
    encoder_name = "resnet34",
    in_channels = 1,
    classes = 3
)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)

### Training The Model

In [ ]:
# best_loss = float("inf")

# num_epochs = 25

# for epoch in range(num_epochs):

#     model.train()
#     train_loss = 0

#     for imgs , masks in train_loader:
#         imgs = imgs.to(device)
#         mask = mask.to(device)

#         outputs = model(imgs)
#         loss = loss_fn(outputs, masks)

#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         train_loss += loss.item()
    
#     #Validation
#     model.eval()
#     val_loss = 0

#     with torch.no_grad():
#         for imgs, mask in val_loader:
#             imgs = imgs.to(device)
#             mask = masks.to(device)

#             outputs = model(imgs)
#             loss = loss_fn(outputs, masks)

#             val_loss += loss.item()
    
#     print(f"Epoch {epoch+1}/{num_epochs} | Train Loss : {train_loss : .4f} | Val Loss : {val_loss : .4f}")

#     # Save the best model
#     if val_loss < best_loss:
#         best_loss = val_loss
#         torch.save(model.state_dict(), "../models/best_model.pth")
#         print("Best model saved")

In [ ]:
# To detect if GPU is available
import torch
print(torch.cuda.is_available())

False


In [14]:
import torch

print("Torch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

Torch version: 2.11.0+cpu
CUDA Available: False
